In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("All tools loaded. Ready to read customer feelings.")

In [ ]:
np.random.seed(42)

negative_templates = [
    "The network is terrible in {place}, calls drop every day",
    "I have been overcharged again on my bill, this is unacceptable",
    "No internet for three days in {place} and no one helped me",
    "Worst customer service ever, I waited two hours and got nothing",
    "My data ran out too fast, I am very disappointed with Omantel",
    "The signal is awful and I am thinking of switching to a competitor",
    "I am so frustrated, my SIM activation failed twice",
    "Slow internet, expensive plan, terrible experience overall",
]
positive_templates = [
    "The new 5G in {place} is amazing, very happy with the speed",
    "Great customer service today, the agent solved my problem quickly",
    "I love the new eSIM, activation was easy and fast",
    "Excellent coverage during my trip to {place}, very satisfied",
    "Best network in Oman, thank you Omantel for the great service",
    "The app is wonderful and my bill was correct this month",
    "Fast internet and a fair price, I am really pleased",
    "Fantastic experience, the National Day offer was a great deal",
]
neutral_templates = [
    "I want to know my current data balance for this month",
    "Please tell me how to activate international roaming for {place}",
    "What time does the Omantel store in {place} open",
    "I need to update the address on my account",
    "Can you explain the charges on my last invoice",
    "How do I transfer my number to a new SIM card",
    "Is the eSIM National Day offer still available",
    "I would like to change my monthly plan",
]
places = ["Muscat", "Salalah", "Sohar", "Nizwa", "Sur"]

rows = []


def fill(t):
    return t.replace("{place}", np.random.choice(places))


for _ in range(120):
    rows.append((fill(np.random.choice(negative_templates)), "Negative"))
for _ in range(90):
    rows.append((fill(np.random.choice(positive_templates)), "Positive"))
for _ in range(90):
    rows.append((fill(np.random.choice(neutral_templates)), "Neutral"))

df = pd.DataFrame(rows, columns=["complaint_text", "true_feeling"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.insert(0, "ticket_id", [f"TKT{50000 + i}" for i in range(len(df))])
print(f"Created {len(df)} customer messages.")
print(df["true_feeling"].value_counts())

# --- save the raw message database as a CSV in the notebook's folder --- df.to_csv("omantel_customer_messages.csv", index=False) print("\nSaved to omantel_customer_messages.csv")


In [ ]:
df.head(10)

In [ ]:
positive_words = {
    "amazing",
    "happy",
    "great",
    "love",
    "easy",
    "fast",
    "excellent",
    "satisfied",
    "best",
    "thank",
    "wonderful",
    "correct",
    "pleased",
    "fantastic",
    "good",
    "fair",
    "quickly",
    "solved",
}
negative_words = {
    "terrible",
    "drop",
    "overcharged",
    "unacceptable",
    "no",
    "awful",
    "worst",
    "waited",
    "nothing",
    "disappointed",
    "frustrated",
    "failed",
    "slow",
    "expensive",
    "switching",
    "competitor",
    "bad",
    "never",
}


def score_message(text):
    words = text.lower().replace(",", " ").split()
    pos = sum(w in positive_words for w in words)
    neg = sum(w in negative_words for w in words)
    if pos > neg:
        return "Positive", pos - neg
    if neg > pos:
        return "Negative", pos - neg
    return "Neutral", 0


df[["predicted_feeling", "score"]] = df["complaint_text"].apply(
    lambda t: pd.Series(score_message(t))
)
print("Sentiment engine applied to every message.")
print(df[["complaint_text", "predicted_feeling"]].head(6).to_string())


In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(df["true_feeling"], df["predicted_feeling"]) * 100
print(f"Sentiment accuracy: {acc:.1f}%")

In [ ]:
counts = df["predicted_feeling"].value_counts()
print("Customer mood right now:")
print(counts)
neg_share = counts.get("Negative", 0) / len(df) * 100
print(f"\nNegative share: {neg_share:.1f}% of all messages")

counts.plot(kind="bar", color=["#C0392B", "#7F8C8D", "#1E7A46"], figsize=(6, 4), rot=0)
plt.title("Omantel customer mood — this week")
plt.ylabel("Number of messages")
plt.show()


In [ ]:
angry = df[df["predicted_feeling"] == "Negative"].sort_values("score")
print(f"MOST NEGATIVE messages to action first ({len(angry)} total):")
print(angry[["ticket_id", "complaint_text", "score"]].head(8).to_string())

angry.to_csv("omantel_negative_tickets.csv", index=False)
print("\nSaved angry list to omantel_negative_tickets.csv")


In [ ]:
from transformers import pipeline

# Loads a real deep-learning sentiment model (DistilBERT).
# Pre-installed by IT; loads instantly from local cache in class.
sentiment_ai = pipeline(
    "sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Deep-learning model loaded. Let's read some messages.")

# Try it on three of our messages
samples = [
    "The signal is awful and I am thinking of switching to a competitor",
    "The new 5G in Sur is amazing, very happy with the speed",
    "I need to update the address on my account",
]
for msg in samples:
    result = sentiment_ai(msg)[0]
    print(f"{result['label']:9} ({result['score'] * 100:.0f}%)  <-  {msg}")


In [ ]:
hard_messages = [
    "The network is not bad actually",  # "not bad"
    "The staff were not helpful at all",  # negation
    "I was worried about my bill but it turned out fine",  # context
    "Nothing about this service is good",  # double-negative
    "Oh brilliant, no signal AGAIN. I just love paying for nothing",  # sarcasm
]

print(f"{'MESSAGE':52} {'OUR COUNTER':12} {'TRANSFORMER':12}")
print("-" * 80)
for msg in hard_messages:
    ours, _ = score_message(msg)
    ai = sentiment_ai(msg)[0]["label"].title()
    print(f"{msg[:50]:52} {ours:12} {ai:12}")
